# Kazakh ASR audit in Colab

This notebook is a safe, step-by-step workflow for running the ASR audit project in Google Colab without stressing your local laptop.

Checklist:
- install dependencies
- set Hugging Face cache in /content
- run a tiny smoke test
- run real data preparation
- transcribe with Whisper
- evaluate results
- save outputs to Google Drive if needed


In [ ]:
# 1) Mount Drive (optional but recommended)
from google.colab import drive

drive.mount('/content/drive')
print('Drive mounted.')


In [1]:
# 2) Get the project into the runtime
#
# Repo is private, so an unauthenticated clone fails. Paste a GitHub token
# when prompted (github.com/settings/tokens -> Generate new token (classic)
# -> scope "repo"). getpass hides the input and it is never written to disk
# or to this notebook file.

import os
from getpass import getpass

REPO_URL = 'https://github.com/assemqb/audit.git'
project_dir = '/content/kk-asr-audit'

if not os.path.exists(project_dir):
    token = getpass('GitHub token (repo scope): ')
    auth_url = REPO_URL.replace('https://', f'https://{token}@')
    !git clone "$auth_url" "$project_dir"
    del token, auth_url  # don't keep the token sitting around in memory

os.chdir(project_dir)
print('Project root:', project_dir)


GitHub token (repo scope): ··········
Cloning into '/content/kk-asr-audit'...
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (18/18), done.
remote: Total 23 (delta 8), reused 20 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (23/23), 16.39 KiB | 5.46 MiB/s, done.
Resolving deltas: 100% (8/8), done.
Project root: /content/kk-asr-audit


In [2]:
# 3) Install dependencies in a safe, reproducible order
!pip install -U pip
!pip install -r requirements.txt

# torch/torchcodec are only used to decode FLEURS audio in prepare_data.py
# (a CPU-side task, nothing to do with Whisper's own GPU speed, which comes
# from ctranslate2). Installing from the CPU wheel index avoids a GPU build
# that expects a matching CUDA nvrtc runtime and fails with
# "libnvrtc.so.13: cannot open shared object file" if the versions don't line up.
!pip install --index-url https://download.pytorch.org/whl/cpu torch torchcodec

import os

# Same path prepare_data.py computes on its own (repo_root/.hf_cache) so every
# step shares one cache instead of downloading models/data twice.
hf_cache = os.path.join(project_dir, '.hf_cache')
os.environ['HF_HOME'] = hf_cache
os.environ['HUGGINGFACE_HUB_CACHE'] = os.path.join(hf_cache, 'hub')
os.environ['HF_DATASETS_CACHE'] = os.path.join(hf_cache, 'datasets')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(hf_cache, 'transformers')
os.environ['HF_HUB_DISABLE_XET'] = '1'

print('Dependencies installed and cache configured at', hf_cache)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 73.8 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 48.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 67.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 109.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 56.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 62.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [faster-whisper]
Looking in indexes: https://download.pytorch.org/whl/cpu
Dependencies installed and cache configured at /content/kk-asr-audit/.hf_cache


In [3]:
# 4) Safe smoke test: tiny data, tiny run
!python src/prepare_data.py --minutes 1 --split test --limit 1
!python src/transcribe.py --model small --lang kk --device auto --compute_type int8
!python src/evaluate.py --hyp results/hyp_small_kk.tsv
print('Smoke test finished.')


README.md: 100% 386k/386k [00:00<00:00, 166MB/s]
Записей: 1
Суммарная длительность: 0.25 мин
Эталоны: data/refs.tsv

Готово: results/hyp_small_kk.tsv
Аудио: 0.25 мин, время работы: 0.02 мин
RTF: 0.079
{
  "n_utt": 1,
  "audio_min": 0.25,
  "WER_raw": 0.7,
  "WER_norm": 0.6667,
  "WER_fold": 0.619,
  "WER_stem": 0.6667,
  "CER_norm": 0.1522,
  "delta_normalization": 0.0333,
  "delta_kk_graphemes": 0.0477,
  "delta_morphology": 0.0
}

Типы ошибок:
      8   57.1%  близкая форма (фонетика/опечатка)
      2   14.3%  пропуск слова
      2   14.3%  полная замена слова
      1    7.1%  уход в русскую графику
      1    7.1%  спецбуквы (ә/ө/ұ/ү/і/ң/қ/ғ)

Топ подмен символов:
      3  ∅ -> ы
      2  ∅ -> н
      2  о -> а
      2  ∅ -> і
      1  ∅ -> т
      1  ∅ -> ң
      1  ∅ -> д
      1  ∅ -> и
      1  ∅ -> м
      1  ∅ -> а
      1  ∅ -> с
      1  а -> я
      1  ң -> н
      1  х -> қ
      1  т -> ң

Кандидатов в галлюцинации: 0
Определённый язык: [('kk', 1)]
Smoke test finished.


In [4]:
# 5) Full run for the actual audit
# --device auto picks the Colab GPU automatically when the runtime has one
# (Runtime > Change runtime type > T4 GPU), otherwise falls back to CPU.
!python src/prepare_data.py --minutes 12 --split test
!python src/transcribe.py --model large-v3 --lang kk --device auto --compute_type int8_float16
!python src/transcribe.py --model small --lang kk --device auto --compute_type int8
!python src/transcribe.py --model large-v3 --lang none --device auto --compute_type int8_float16
!python src/evaluate.py --hyp results/hyp_large-v3_kk.tsv
print('Full audit finished.')


Записей: 43
Суммарная длительность: 12.05 мин
Эталоны: data/refs.tsv
  20/43
  40/43

Готово: results/hyp_large-v3_kk.tsv
Аудио: 12.05 мин, время работы: 1.40 мин
RTF: 0.116
  20/43
  40/43

Готово: results/hyp_small_kk.tsv
Аудио: 12.05 мин, время работы: 0.53 мин
RTF: 0.044
  20/43
  40/43

Готово: results/hyp_large-v3_none.tsv
Аудио: 12.05 мин, время работы: 1.68 мин
RTF: 0.140
{
  "n_utt": 43,
  "audio_min": 12.05,
  "WER_raw": 0.3725,
  "WER_norm": 0.3055,
  "WER_fold": 0.2924,
  "WER_stem": 0.2833,
  "CER_norm": 0.0617,
  "delta_normalization": 0.067,
  "delta_kk_graphemes": 0.0131,
  "delta_morphology": 0.0222
}

Типы ошибок:
    140   59.8%  близкая форма (фонетика/опечатка)
     31   13.2%  полная замена слова
     18    7.7%  вставка слова
     15    6.4%  пропуск слова
     12    5.1%  аффикс (морфология)
     10    4.3%  спецбуквы (ә/ө/ұ/ү/і/ң/қ/ғ)
      6    2.6%  уход в русскую графику
      1    0.4%  числа/нормализация
      1    0.4%  спецбуквы + аффикс

Топ подмен симв

In [1]:
# 6) Optional: save results to Drive
# Use if you want the outputs to survive session restarts.

drive_dir = '/content/drive/MyDrive/kk-asr-audit'
!mkdir -p "$drive_dir"
!cp -r /content/kk-asr-audit/data "$drive_dir/"
!cp -r /content/kk-asr-audit/results "$drive_dir/"
print('Results copied to Drive.')


cp: cannot stat '/content/kk-asr-audit/data': No such file or directory
cp: cannot stat '/content/kk-asr-audit/results': No such file or directory
Results copied to Drive.
